# 04 - Mesh tallies and weight windows

Solutions of the exercises. The setup cell below rebuilds the few objects they need, so this
notebook runs on its own.

In [ ]:
# Run this cell first. It sets up the paths and the objects used by the exercises.
from pathlib import Path

import numpy as np
import pyvista as pv

from f4enix.input.ww_gvr import WW
from f4enix.output.meshtal.meshtal import Meshtal
from f4enix.output.plotter import Atlas, MeshPlotter

DATA = Path("..") / "data"        # the MCNP files, read-only
OUT = Path("..") / "outputs"      # everything we produce goes here
OUT.mkdir(exist_ok=True)

# a slice through an empty region should give an empty picture, not an error
pv.global_theme.allow_empty_mesh = True

meshtal = Meshtal(DATA / "iter_1d" / "meshtal")
meshtal.readMesh()


def max_ratio(weight_window):
    return max(
        np.max(array)
        for by_energy in weight_window.ratios.values()
        for array in by_energy.values()
    )


print("mesh tallies in the file:", meshtal.mesh_ids)

---

# Exercises

## Exercise 1 - a photon GVR

Build the equivalent weight window for photons, out of mesh tally **1024**:

1. create the GVR with a maximum splitting ratio of 5;
2. soften it with a factor of **0.6**;
3. mitigate the long histories with a maximum ratio of **10**;
4. write `outputs/gvr_photon` (the MCNP file) and `outputs/gvr_photon.vts` (for ParaView).

**Hints**
* `WW.create_gvr_from_meshtally_file(path, tally_num=..., maximum_splitting_ratio=...)`;
* the three operations are methods on the object, applied in place;
* `write_to_ww_file` and `export_as_vtk`.

In [ ]:
# 1: same call as before, on the photon mesh tally
photon_gvr = WW.create_gvr_from_meshtally_file(
    DATA / "iter_1d" / "meshtal",
    tally_num=1024,
    maximum_splitting_ratio=5.0,
)

# 2: flatten the distribution a bit
photon_gvr.soften(0.6)

# 3: kill the voxels that would cause long histories
photon_gvr.mitigate_long_histories(max_ratio=10.0)

# 4: one file for MCNP, one for ParaView
photon_gvr.write_to_ww_file(OUT / "gvr_photon")
photon_gvr.export_as_vtk(OUT / "gvr_photon.vts")

print(photon_gvr.info)

In [ ]:
# --- self-check, do not modify ---
assert (OUT / "gvr_photon").is_file(), "the weight window file was not written"
assert (OUT / "gvr_photon.vts").is_file(), "the VTK file was not written"
assert max_ratio(photon_gvr) <= 10.0 + 1e-6, (
    f"the maximum ratio is still {max_ratio(photon_gvr):.3e} - did you mitigate?"
)
print("Exercise 1 correct")